In [ ]:
!pip install scanpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.2/174.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 25.0 MB/s eta 0:00:00


In [ ]:
!pip install scikit-misc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 6.5 MB/s eta 0:00:00


In [ ]:
!pip -q install scanpy scikit-learn pandas numpy

import os, subprocess
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.ensemble import RandomForestRegressor

COUNTS_PATH = "/content/Combined_UMI_table.txt"   # genes x cells
TF_PATH     = "/content/hs_tfs_hg38.txt"
TARGETS     = ["PRM1", "PRM2"]

# QC
MIN_COUNTS = 1000
MIN_GENES  = 200
MAX_PCT_MT = 15
MIN_CELLS_PER_GENE = 10

# HVG
N_TOP_HVG = 3000
HVG_FLAVOR = "seurat_v3"

# GENIE3 core defaults
NTREES = 1000
K = "sqrt"
RANDOM_STATE = 0
N_JOBS = -1

# Bootstrap stability
N_BOOT = 30
CELL_FRAC = 0.75
NTREES_BOOT = 500

TOP_K = 20


df = pd.read_csv(COUNTS_PATH, sep="\t", index_col=0)
adata = sc.AnnData(df.T)
adata.var_names = adata.var_names.astype(str).str.strip()
adata.var_names_make_unique()

adata.layers["counts"] = adata.X.copy()

# QC
adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)

adata = adata[
    (adata.obs["total_counts"] >= MIN_COUNTS)
    & (adata.obs["n_genes_by_counts"] >= MIN_GENES)
    & (adata.obs["pct_counts_mt"] <= MAX_PCT_MT),
    :
].copy()

sc.pp.filter_genes(adata, min_cells=MIN_CELLS_PER_GENE)

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=N_TOP_HVG,
    flavor=HVG_FLAVOR,
    layer="counts",
    subset=False
)
hvg_set = set(adata.var_names[adata.var["highly_variable"]])

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

if not os.path.exists(TF_PATH):
    subprocess.run(
        ["bash", "-lc", f"wget -q https://resources.aertslab.org/cistarget/tf_lists/allTFs_hg38.txt -O {TF_PATH}"],
        check=True
    )

tf_list = (
    pd.read_csv(TF_PATH, header=None)[0]
    .astype(str).str.strip().tolist()
)

missing_targets = [g for g in TARGETS if g not in adata.var_names]
if missing_targets:
    raise ValueError(f"Targets not found after QC: {missing_targets}")

genes_in_data = set(adata.var_names)
regulators = sorted((set(tf_list) & hvg_set) & genes_in_data)

print(f"Cells: {adata.n_obs}")
print(f"Genes after QC: {adata.n_vars}")
print(f"HVGs: {len(hvg_set)}")
print(f"Strict regulators (TF ∩ HVG): {len(regulators)}")
print(f"Targets: {TARGETS}")

if len(regulators) < 10:
    raise ValueError("Too few TF∩HVG regulators found. Consider increasing N_TOP_HVG or relaxing criteria.")

keep_genes = sorted(set(regulators).union(TARGETS))
adata_m = adata[:, keep_genes].copy()

expr = pd.DataFrame(
    adata_m.X.A if hasattr(adata_m.X, "A") else adata_m.X,
    index=adata_m.obs_names,
    columns=adata_m.var_names
).astype(np.float32)

def genie3_target(expr_df, regulators, target, ntrees, random_state):
    regs = [r for r in regulators if r != target and r in expr_df.columns]
    X = expr_df[regs].to_numpy(dtype=np.float32)
    y = expr_df[target].to_numpy(dtype=np.float32)

    y_std = float(np.std(y))
    if y_std == 0.0:
        importances = np.zeros(len(regs), dtype=np.float32)
    else:
        y = y / y_std

        if K == "all" or (isinstance(K, int) and K >= len(regs)):
            max_features = 1.0
        else:
            max_features = K

        model = RandomForestRegressor(
            n_estimators=ntrees,
            max_features=max_features,
            random_state=random_state,
            n_jobs=N_JOBS
        )
        model.fit(X, y)
        importances = model.feature_importances_.astype(np.float32)

    out = pd.DataFrame({"TF": regs, "importance": importances})
    out = out.sort_values("importance", ascending=False).reset_index(drop=True)
    out.insert(1, "target", target)
    return out

results = pd.concat(
    [genie3_target(expr, regulators, t, ntrees=NTREES, random_state=RANDOM_STATE) for t in TARGETS],
    ignore_index=True
)

print("\n=== Top 20 TF -> PRM1 (strict TF∩HVG, all cells) ===")
print(results[results["target"]=="PRM1"].head(TOP_K).to_string(index=False))

print("\n=== Top 20 TF -> PRM2 (strict TF∩HVG, all cells) ===")
print(results[results["target"]=="PRM2"].head(TOP_K).to_string(index=False))

# Bootstrap stability
rng = np.random.default_rng(0)
boot_counts = {t: pd.Series(0, index=regulators, dtype=int) for t in TARGETS}

if N_BOOT > 0:
    cell_ids = expr.index.to_numpy()
    n_sub = int(CELL_FRAC * len(cell_ids))

    for b in range(N_BOOT):
        subs = rng.choice(cell_ids, size=n_sub, replace=False)
        expr_b = expr.loc[subs]

        for t in TARGETS:
            df_imp = genie3_target(expr_b, regulators, t, ntrees=NTREES_BOOT, random_state=b)
            top_tfs = df_imp["TF"].head(TOP_K).tolist()
            boot_counts[t].loc[top_tfs] += 1

        if (b+1) % 5 == 0 or b == 0:
            print(f"Bootstrap {b+1}/{N_BOOT} done")

    stable = {}
    for t in TARGETS:
        stable[t] = (
            boot_counts[t]
            .sort_values(ascending=False)
            .rename("count")
            .reset_index()
            .rename(columns={"index":"TF"})
        )
        print(f"\n=== Stable TF→{t}: Top 20 by Top-{TOP_K} frequency (n_boot={N_BOOT}) ===")
        print(stable[t].head(20).to_string(index=False))

    results.to_csv("/content/GENIE3_strict_TF_HVG_all_edges.tsv", sep="\t", index=False)
    results[results["target"]=="PRM1"].head(50).to_csv("/content/PRM1_top50_GENIE3_strict_TF_HVG.tsv", sep="\t", index=False)
    results[results["target"]=="PRM2"].head(50).to_csv("/content/PRM2_top50_GENIE3_strict_TF_HVG.tsv", sep="\t", index=False)

    stable["PRM1"].to_csv("/content/PRM1_stability_counts_GENIE3_strict_TF_HVG.tsv", sep="\t", index=False)
    stable["PRM2"].to_csv("/content/PRM2_stability_counts_GENIE3_strict_TF_HVG.tsv", sep="\t", index=False)

    print("\nSaved:")
    print(" - /content/GENIE3_strict_TF_HVG_all_edges.tsv")
    print(" - /content/PRM1_top50_GENIE3_strict_TF_HVG.tsv")
    print(" - /content/PRM2_top50_GENIE3_strict_TF_HVG.tsv")
    print(" - /content/PRM1_stability_counts_GENIE3_strict_TF_HVG.tsv")
    print(" - /content/PRM2_stability_counts_GENIE3_strict_TF_HVG.tsv")


Cells: 6033
Genes after QC: 24975
HVGs: 3000
Strict regulators (TF ∩ HVG): 194
Targets: ['PRM1', 'PRM2']

=== Top 20 TF -> PRM1 (strict TF∩HVG, all cells) ===
    TF target  importance
 HMGB4   PRM1    0.156049
 HMGB1   PRM1    0.068003
 H2AFZ   PRM1    0.049490
 RPS4X   PRM1    0.049354
ZNF683   PRM1    0.042874
   RAN   PRM1    0.042781
 HSPA5   PRM1    0.038409
  SMC3   PRM1    0.038072
 HMGB2   PRM1    0.031683
  ILF2   PRM1    0.023610
 CEBPD   PRM1    0.019793
  CD59   PRM1    0.018876
   UBB   PRM1    0.016759
  YBX1   PRM1    0.016614
GTF2A2   PRM1    0.016383
 TBPL1   PRM1    0.015921
 TFDP2   PRM1    0.014720
   JUN   PRM1    0.014480
 NR2F2   PRM1    0.013975
  CREM   PRM1    0.010474

=== Top 20 TF -> PRM2 (strict TF∩HVG, all cells) ===
    TF target  importance
 HMGB4   PRM2    0.107833
 HMGB1   PRM2    0.071810
 RPS4X   PRM2    0.054402
 H2AFZ   PRM2    0.053042
  SMC3   PRM2    0.041127
   RAN   PRM2    0.040313
 HSPA5   PRM2    0.039598
 HMGB2   PRM2    0.034129
ZNF683 